In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, f1_score, roc_curve, auc, classification_report
from tabulate import tabulate


In [ ]:
# Load the data
secom_df = pd.read_csv('uci-secom.csv')
secom_df = secom_df.iloc[:,1:] # Drop the first column (time)

In [ ]:
secom_df.head()

In [ ]:
secom_df.info()

In [ ]:
# Try different cleaning strategies for data cleaning process
secom_df_a = secom_df.copy().fillna(0)
secom_df_b = secom_df_a.copy()
single_value_columns = secom_df_b.columns[secom_df_b.nunique() == 1]
secom_df_b = secom_df_b.drop(columns=single_value_columns)

print('Columns dropped: ' + str(single_value_columns))

secom_df_c = secom_df_b.copy()
correlation_matrix = secom_df_c.corr().abs()

columns_to_drop = set()
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if correlation_matrix.iloc[i, j] > 0.8:
            colname = correlation_matrix.columns[i]
            if colname not in columns_to_drop:
                columns_to_drop.add(colname)

secom_df_c = secom_df_c.drop(columns=list(columns_to_drop))

print('Columns dropped: ' + str(columns_to_drop))

secom_df_d = secom_df.copy().fillna(secom_df.mean())
secom_df_e = secom_df_d.copy()
single_value_columns = secom_df_e.columns[secom_df_e.nunique() == 1]
secom_df_e = secom_df_e.drop(columns=single_value_columns)

print('Columns dropped: ' + str(single_value_columns))

secom_df_f = secom_df_e.copy()
correlation_matrix = secom_df_f.corr().abs()

columns_to_drop = set()
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if correlation_matrix.iloc[i, j] > 0.8:
            colname = correlation_matrix.columns[i]
            if colname not in columns_to_drop:
                columns_to_drop.add(colname)

secom_df_f = secom_df_f.drop(columns=list(columns_to_drop))

print('Columns dropped: ' + str(columns_to_drop))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score
from sklearn.model_selection import train_test_split

# Initialize the classifier
rf_classifier = RandomForestClassifier(
    n_estimators=20,
    random_state=42,
    max_depth=20
)

def train_and_evaluate_rf(data, model_name):
    train, test = train_test_split(data, test_size=0.2, random_state=42)

    X_train = train.iloc[:, :-1]
    y_train = train.iloc[:, -1]

    X_test = test.iloc[:, :-1]
    y_test = test.iloc[:, -1]

    # Train the model
    rf_classifier.fit(X_train, y_train)

    # Make predictions
    y_pred = rf_classifier.predict(X_test)

    # Calculate metrics
    cm = confusion_matrix(y_test, y_pred)
    accuracy = accuracy_score(y_test, y_pred)
    sensitivity = recall_score(y_test, y_pred)
    specificity = cm[0,0] / (cm[0,0] + cm[0,1])
    f1 = f1_score(y_test, y_pred)

    print(f"\nResults for Model {model_name}:")
    print("Confusion Matrix:")
    print(cm)
    print(f"\nAccuracy: {accuracy:.10f}")
    print(f"Sensitivity: {sensitivity:.10f}")
    print(f"Specificity: {specificity:.10f}")
    print(f"F1 Score: {f1:.10f}")
    print("-" * 50)

    return rf_classifier, y_pred, train, test, name

# Create a list of tuples instead of a set
models = [
    (secom_df_a, 'a'),
    (secom_df_b, 'b'),
    (secom_df_c, 'c'),
    (secom_df_d, 'd'),
    (secom_df_e, 'e'),
    (secom_df_f, 'f')
]

# Train and evaluate each model
results = {}
for data, name in models:  # Remove the parentheses after models
    model, predictions, train, test, name = train_and_evaluate_rf(data, name)
    results[name] = {
        'name': name,
        'accuracy': accuracy_score(test.iloc[:, -1], predictions),
        'sensitivity': recall_score(test.iloc[:, -1], predictions),
        'specificity': confusion_matrix(test.iloc[:, -1], predictions)[0,0] / (confusion_matrix(test.iloc[:, -1], predictions)[0,0] + confusion_matrix(test.iloc[:, -1], predictions)[0,1]),
        'f1_score': f1_score(test.iloc[:, -1], predictions),
        'model': model,
        'predictions': predictions,
        'train': train,
        'test': test
    }

In [ ]:
from tabulate import tabulate

# Create a list of lists for the table data
table_data = []
for name in results:
    r = results[name]
    table_data.append([
        f"Model {name}",
        f"{r['accuracy']:.10f}",
        f"{r['sensitivity']:.4f}",
        f"{r['specificity']:.10f}",
        f"{r['f1_score']:.4f}"
    ])

# Create and print the table
headers = ["Model", "Accuracy", "Sensitivity", "Specificity", "F1 Score"]
print("\nComparison of all models:")
print(tabulate(table_data, headers=headers, tablefmt="grid"))

In [ ]:
# Create a list of tuples instead of a set
models = [
    (secom_df_a, 'NaN to 0'),
    (secom_df_b, 'Nan to 0, remove single value col'),
    (secom_df_c, 'Nan to 0, remove single value col and col with r>0.8'),
    (secom_df_d, 'NaN to mean col'),
    (secom_df_e, 'Nan to mean col, remove single value col'),
    (secom_df_f, 'Nan to mean col, remove single value col and col with r>0.8')
]



In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split

def plot_roc_curves(datasets):
    plt.figure(figsize=(10, 8))

    # Store AUC scores
    auc_scores = {}

    # Plot ROC curve for each model
    for name, data in datasets.items():
        # Split data
        X = data.iloc[:, :-1]
        y = data.iloc[:, -1]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # Train model
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(X_train, y_train)

        # Get probabilities for ROC curve
        y_prob = rf.predict_proba(X_test)[:, 1]

        # Calculate ROC curve and AUC
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        roc_auc = auc(fpr, tpr)
        auc_scores[name] = roc_auc

        # Plot ROC curve
        plt.plot(fpr, tpr, label=f'Model {name} (AUC = {roc_auc:.3f})')

    # Add diagonal line (random classifier)
    plt.plot([0, 1], [0, 1], 'k--', label='Random')

    # Customize plot
    plt.xlabel('False Positive Rate (1 - Specificity)')
    plt.ylabel('True Positive Rate (Sensitivity)')
    plt.title('ROC Curves for All Models')
    plt.legend(loc='lower right')
    plt.grid(True)

    # Print AUC scores in a table
    print("\nAUC Scores:")
    table_data = [[f"Model {name}", f"{score:.10f}"] for name, score in auc_scores.items()]
    print(tabulate(table_data, headers=["Model", "AUC Score"], tablefmt="grid"))

    plt.show()
    return auc_scores

# Dictionary of datasets
datasets = {
    'a': secom_df_a,
    'b': secom_df_b,
    'c': secom_df_c,
    'd': secom_df_d,
    'e': secom_df_e,
    'f': secom_df_f
}

# Plot ROC curves and get AUC scores
auc_scores = plot_roc_curves(datasets)

In [ ]:
import pandas as pd
from tabulate import tabulate

def show_feature_importance(results, model_name):
    try:
        # Get model and feature importances
        model = results[model_name]['model']

        # Get feature names and importances
        features = list(results[model_name]['train'].iloc[:, :-1].columns)
        importances = list(model.feature_importances_)

        # Create dictionary of feature importances
        importance_dict = dict(zip(features, importances))

        # Sort by importance and get top 20
        sorted_importances = dict(sorted(importance_dict.items(),
                                       key=lambda x: x[1],
                                       reverse=True)[:20])

        # Create lists for table
        top_features = list(sorted_importances.keys())
        top_importance_values = list(sorted_importances.values())

        # Print table
        print(f"\nTop 20 Feature Importance for Model {model_name}:")
        table_data = list(zip(top_features, top_importance_values))
        print(tabulate(table_data,
                      headers=['Feature', 'Importance'],
                      tablefmt="grid",
                      floatfmt=".10f"))

        return sorted_importances

    except Exception as e:
        print(f"Error processing model {model_name}: {str(e)}")
        return None

# Show feature importance for each model
importance_results = {}
for model_name in results.keys():
    importance_results[model_name] = show_feature_importance(results, model_name)

In [ ]:
def compare_shared_features(importance_results):
    # Create sets of top features for each model
    model_features = {
        model: set(importances.keys())
        for model, importances in importance_results.items()
        if importances is not None
    }

    # Create comparison table for all pairs of models
    models = list(model_features.keys())
    comparison_data = []

    for i in range(len(models)):
        for j in range(i+1, len(models)):
            model1, model2 = models[i], models[j]
            shared_features = model_features[model1].intersection(model_features[model2])

            # Get the shared features with their importances from both models
            shared_feature_details = []
            for feature in shared_features:
                importance1 = importance_results[model1][feature]
                importance2 = importance_results[model2][feature]
                shared_feature_details.append([
                    feature,
                    importance1,
                    importance2
                ])

            # Sort by average importance
            shared_feature_details.sort(key=lambda x: (x[1] + x[2])/2, reverse=True)

            print(f"\nShared Features between Model {model1} and Model {model2}")
            print(f"Number of shared features: {len(shared_features)}")

            if shared_features:
                print(tabulate(shared_feature_details,
                             headers=[
                                 'Feature',
                                 f'Importance in {model1}',
                                 f'Importance in {model2}'
                             ],
                             tablefmt="grid",
                             floatfmt=".10f"))

            # Add to comparison data
            comparison_data.append([
                f"{model1}-{model2}",
                len(shared_features)
            ])

    # Print summary table of number of shared features
    print("\nSummary of Shared Features:")
    print(tabulate(comparison_data,
                  headers=['Model Pair', 'Number of Shared Features'],
                  tablefmt="grid"))

# Run the comparison
compare_shared_features(importance_results)